# Stockfish Eval Debug Colab

Focused Colab notebook for debugging Stockfish evaluation against a checkpoint stored on Google Drive.

This notebook reuses the repo/bootstrap flow from `chess_model_run_git.ipynb`, but replaces training with staged eval diagnostics:
- mount Drive and clone the repo
- install Colab dependencies and Stockfish
- discover candidate checkpoints on Drive
- load either a Lightning checkpoint or a DM-port base checkpoint
- run `ReasoningEvaluator.single_evaluation()` with a small game count
- print raw results, PGNs, and full tracebacks on failure


In [ ]:
#@title Runtime Parameters
REPO_URL = "https://github.com/noamdwc/grpo_chess.git"  #@param {type:"string"}
REPO_REF = "search_refactor"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/data/grpo-chess"  #@param {type:"string"}
BASE_CHECKPOINT_PATH = "/content/drive/MyDrive/data/grpo-chess/base/9M.pt"  #@param {type:"string"}
CHECKPOINT_ROOT = "/content/drive/MyDrive/data/grpo-chess/runs"  #@param {type:"string"}
CHECKPOINT_FILTER = ""  #@param {type:"string"}
CHECKPOINT_INDEX = 0  #@param {type:"integer"}
CONFIG_NAME = "grpo_colab_main.yaml"  #@param {type:"string"}
STOCKFISH_PATH = ""  #@param {type:"string"}
EVAL_GAMES = 2  #@param {type:"integer"}
MAX_PLIES = 120  #@param {type:"integer"}
OPENING_PLIES = 4  #@param {type:"integer"}
STOCKFISH_MOVETIME_MS = 20  #@param {type:"integer"}
STOCKFISH_SKILL_LEVEL = 2  #@param {type:"integer"}
THINK_TOKENS = 12  #@param {type:"integer"}
PRINT_FULL_PGNS = False  #@param {type:"boolean"}


In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    raise RuntimeError("This notebook is intended to run in Google Colab.")

from google.colab import drive
drive.mount("/content/drive")

repo = Path("/content/grpo_chess")
os.chdir("/content")
if repo.exists():
    shutil.rmtree(repo)

!git clone {REPO_URL} /content/grpo_chess
%cd /content/grpo_chess
!git checkout {REPO_REF}
!git submodule update --init --recursive

if str(repo) not in sys.path:
    sys.path.append(str(repo))

drive_root = Path(DRIVE_ROOT)
drive_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
print("Repo:", repo)
print("Drive root:", drive_root)


In [ ]:
import os
import signal
from pathlib import Path

deps_ready = Path("/tmp/grpo_stockfish_eval_debug_deps_ready")
if not deps_ready.exists():
    %pip install -q --upgrade pip setuptools wheel
    !grep -vE '^numpy==' requirements.txt > /tmp/requirements-colab.txt
    %pip install -q -r /tmp/requirements-colab.txt
    !apt-get -qq update
    !apt-get -qq install -y stockfish
    !COLAB=1 bash scripts/setup_distill_deps.sh --checkpoint 9M --skip-checkpoint
    %pip install -q --force-reinstall --no-cache-dir pillow
    deps_ready.touch()
    print("Dependencies installed. Restarting runtime to load fresh binary modules...")
    os.kill(os.getpid(), signal.SIGKILL)
else:
    print("Dependencies already installed for this runtime.")


In [ ]:
from datetime import datetime
from pathlib import Path

checkpoint_root = Path(CHECKPOINT_ROOT).expanduser()
if not checkpoint_root.exists():
    raise FileNotFoundError(f"CHECKPOINT_ROOT does not exist: {checkpoint_root}")

patterns = ("*.ckpt", "*.pt", "*.pth")
candidates = []
for pattern in patterns:
    candidates.extend(checkpoint_root.rglob(pattern))

if CHECKPOINT_FILTER.strip():
    needle = CHECKPOINT_FILTER.strip().lower()
    candidates = [p for p in candidates if needle in str(p).lower()]

candidates = sorted(
    {p.resolve() for p in candidates if p.is_file()},
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if not candidates:
    raise RuntimeError(
        f"No checkpoints found under {checkpoint_root} with filter={CHECKPOINT_FILTER!r}"
    )

print(f"Found {len(candidates)} candidate checkpoints")
for idx, path in enumerate(candidates[:20]):
    stat = path.stat()
    stamp = datetime.fromtimestamp(stat.st_mtime).isoformat(timespec="seconds")
    print(f"[{idx}] {path} | {stat.st_size / 1e6:.1f} MB | mtime={stamp}")

if CHECKPOINT_INDEX < 0 or CHECKPOINT_INDEX >= len(candidates):
    raise IndexError(f"CHECKPOINT_INDEX {CHECKPOINT_INDEX} is out of range for {len(candidates)} candidates")

selected_checkpoint = candidates[CHECKPOINT_INDEX]
base_checkpoint = Path(BASE_CHECKPOINT_PATH).expanduser()
print("Selected checkpoint:", selected_checkpoint)
print("Base checkpoint:", base_checkpoint)


In [ ]:
import traceback
import torch

from src.checkpoint_compat import load_checkpoint_with_compat, load_state_dict_with_checkpoint_compat
from src.configs.config_loader import load_experiment_config
from src.grpo_logic.model import ReasoningGRPOLightningModule
from src.train_self_play import ReasoningEvaluator
from src.chess.stockfish import resolve_stockfish_path


def extract_reasoning_state_dict(checkpoint: dict) -> dict[str, torch.Tensor]:
    if "model_state_dict" in checkpoint:
        return checkpoint["model_state_dict"]
    if "state_dict" in checkpoint:
        raw = checkpoint["state_dict"]
        if any(key.startswith("policy_model.") for key in raw):
            return {key[13:]: value for key, value in raw.items() if key.startswith("policy_model.")}
        if any(key.startswith("model.") for key in raw):
            return {key[6:]: value for key, value in raw.items() if key.startswith("model.")}
        return raw
    return checkpoint


raw_checkpoint = load_checkpoint_with_compat(str(selected_checkpoint), map_location="cpu", weights_only=False)
checkpoint_keys = sorted(raw_checkpoint.keys()) if isinstance(raw_checkpoint, dict) else []
print("Top-level checkpoint keys:", checkpoint_keys)

is_lightning_checkpoint = isinstance(raw_checkpoint, dict) and "pytorch-lightning_version" in raw_checkpoint
is_dm_port_checkpoint = isinstance(raw_checkpoint, dict) and "config" in raw_checkpoint and "state_dict" in raw_checkpoint and not is_lightning_checkpoint

effective_base_checkpoint = selected_checkpoint if is_dm_port_checkpoint else base_checkpoint
if not effective_base_checkpoint.exists():
    raise FileNotFoundError(f"Effective base checkpoint does not exist: {effective_base_checkpoint}")

overrides = {
    "model": {"base_checkpoint": str(effective_base_checkpoint)},
    "rival": {"frozen_dm_9m": {"checkpoint_path": str(effective_base_checkpoint)}},
    "eval": {
        "games": int(EVAL_GAMES),
        "max_plies": int(MAX_PLIES),
        "opening_plies": int(OPENING_PLIES),
    },
    "stockfish": {
        "path": STOCKFISH_PATH.strip() or None,
        "movetime_ms": int(STOCKFISH_MOVETIME_MS),
        "skill_level": int(STOCKFISH_SKILL_LEVEL),
    },
}

cfg = load_experiment_config(CONFIG_NAME, overrides=overrides)
resolved_stockfish = resolve_stockfish_path(cfg.stockfish.path)
print("Checkpoint kind:", "lightning" if is_lightning_checkpoint else "dm_port_base" if is_dm_port_checkpoint else "raw_state_dict")
print("Resolved config:", CONFIG_NAME)
print("Resolved stockfish path:", resolved_stockfish)
print("Eval games:", cfg.eval.games)
print("Think tokens:", THINK_TOKENS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
module = ReasoningGRPOLightningModule(cfg).to(device)
load_missing, load_unexpected = [], []
if not is_dm_port_checkpoint:
    state_dict = extract_reasoning_state_dict(raw_checkpoint)
    load_missing, load_unexpected = load_state_dict_with_checkpoint_compat(
        module.policy_model,
        state_dict,
        checkpoint_path=str(selected_checkpoint),
        strict=False,
    )
model = module.policy_model.eval()

print("Device:", device)
print("Missing keys:", load_missing[:20])
print("Unexpected keys:", load_unexpected[:20])


In [ ]:
import traceback

evaluator = ReasoningEvaluator(
    eval_cfg=cfg.eval,
    stockfish_cfg=cfg.stockfish,
    think_tokens=int(THINK_TOKENS),
)

print("Starting single_evaluation()...")
try:
    with torch.no_grad():
        results, _, pgns = evaluator.single_evaluation(model)
    print("Evaluation succeeded")
    print("Results:", results)
    print("PGN count:", len(pgns))
    if pgns:
        print("First PGN:\n")
        print(pgns[0])
    if PRINT_FULL_PGNS and pgns:
        for idx, pgn in enumerate(pgns):
            print(f"\n===== PGN {idx} =====\n")
            print(pgn)
except Exception as exc:
    print("Evaluation failed:", repr(exc))
    print(traceback.format_exc())
    raise


In [ ]:
from src.evaluator import Evaluator

print("Reproducing callback-style retry loop with safe traceback logging...")
for attempt in range(1, 4):
    try:
        with torch.no_grad():
            retry_results, _, _ = evaluator.single_evaluation(model)
        print(f"Attempt {attempt}: success")
        print(retry_results)
        break
    except Exception as exc:
        print(f"Attempt {attempt}: failure")
        print("Safe message:", Evaluator._safe_exception_message(exc))
        print(Evaluator._safe_traceback_text(exc))
else:
    raise RuntimeError("All 3 callback-style eval attempts failed")
